## Construcción del dataset para el modelo híbrido

En este cuaderno se combinan los datos de TMDB y de MovieLens para construir el dataset que utilizarán los modelos híbridos de recomendación.

In [ ]:
import pandas as pd

Las variables que se combinan de cada fuente son las siguientes.

**TMDB:** id, budget, popularity, vote_average, vote_count, revenue, runtime

**MovieLens:** userId, rating, movieId, year, genres, timestamp

Se cargan los datos de TMDB ya validados en la fase de integridad referencial.

In [82]:
df = pd.read_parquet('../data/02_processed/TMDB_integrity.parquet')
df.head()

,id,title,genres,popularity,overview,tagline,vote_average,vote_count,runtime,budget,revenue,release_date
0,2,Ariel,"[Comedy, Drama, Romance, Crime]",1.3807,A Finnish man goes to the city to find a job a...,,7.121,375,73,0,0,1988-10-21
1,5,Four Rooms,[Comedy],3.6807,It's Ted the Bellhop's first night on the job....,Twelve outrageous guests. Four scandalous requ...,5.904,2847,98,4000000,4257354,1995-12-09
2,6,Judgment Night,"[Action, Crime, Thriller]",1.8049,"Four young friends, while taking a shortcut en...",Don't move. Don't whisper. Don't even breathe.,6.463,377,109,21000000,12136938,1993-10-15
3,11,Star Wars,"[Adventure, Action, Science Fiction]",27.0355,Princess Leia is captured and held hostage by ...,"A long time ago in a galaxy far, far away...",8.205,22361,121,11000000,775398007,1977-05-25
4,12,Finding Nemo,"[Animation, Family, Adventure]",19.5501,"Nemo, an adventurous young clownfish, is unexp...",There are 3.7 trillion fish in the ocean. They...,7.819,20567,100,94000000,940335536,2003-05-30


Se seleccionan las columnas relevantes de TMDB y se ordena el conjunto por número de votos, para poder inspeccionar más fácilmente los casos con menos información.

In [ ]:
campos = ['id','budget','popularity','vote_average','vote_count','revenue', 'runtime','release_date','genres']
df2 = df[campos].sort_values('vote_count')
df2

,id,budget,popularity,vote_average,vote_count,revenue,runtime,release_date,genres
8535,148184,350000,0.1573,0.000,0,0,56,2010-01-01,"[Documentary, History, Thriller]"
9401,363444,0,0.0511,5.000,1,0,90,2005-01-01,[Documentary]
8615,171982,0,0.1295,6.000,1,0,27,2012-10-09,"[Romance, Drama, Comedy]"
8307,103008,0,0.7650,2.000,1,0,85,2011-05-30,[Documentary]
7663,55146,500000,0.2646,7.000,1,0,113,1995-08-29,"[Drama, Thriller, Documentary, Romance]"
...,...,...,...,...,...,...,...,...,...
5325,19995,237000000,47.1913,7.606,34017,2923706026,162,2009-12-16,"[Science Fiction, Action, Adventure]"
98,155,185000000,40.7693,8.531,35844,1004558444,152,2008-07-16,"[Action, Crime, Thriller]"
5776,24428,220000000,52.4967,8.037,38187,1518815515,143,2012-04-25,"[Science Fiction, Action, Adventure]"
6076,27205,160000000,34.3061,8.372,39318,839030630,148,2010-07-15,"[Action, Science Fiction, Adventure]"


In [85]:
df2.info()

<class 'pandas.DataFrame'>
Index: 9615 entries, 8535 to 8560
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            9615 non-null   int64  
 1   budget        9615 non-null   int64  
 2   popularity    9615 non-null   float64
 3   vote_average  9615 non-null   float64
 4   vote_count    9615 non-null   int64  
 5   revenue       9615 non-null   int64  
 6   runtime       9615 non-null   int64  
 7   release_date  9615 non-null   str    
 8   genres        9615 non-null   object 
dtypes: float64(2), int64(5), object(1), str(1)
memory usage: 846.2+ KB


Antes de continuar se revisan las columnas budget, revenue y runtime, ya que pueden contener valores en 0 que en realidad correspondan a datos faltantes.

In [86]:
df2[df2['revenue']==0]

,id,budget,popularity,vote_average,vote_count,revenue,runtime,release_date,genres
8535,148184,350000,0.1573,0.000,0,0,56,2010-01-01,"[Documentary, History, Thriller]"
9401,363444,0,0.0511,5.000,1,0,90,2005-01-01,[Documentary]
8615,171982,0,0.1295,6.000,1,0,27,2012-10-09,"[Romance, Drama, Comedy]"
8307,103008,0,0.7650,2.000,1,0,85,2011-05-30,[Documentary]
7663,55146,500000,0.2646,7.000,1,0,113,1995-08-29,"[Drama, Thriller, Documentary, Romance]"
...,...,...,...,...,...,...,...,...,...
9459,387426,50000000,4.3647,7.333,4542,0,120,2017-06-28,"[Adventure, Drama, Science Fiction]"
8498,138697,3000000,6.4698,6.014,4580,0,90,2013-09-12,"[Romance, Comedy, Drama]"
2472,9732,0,8.0736,6.939,4673,0,81,1998-10-24,"[Adventure, Animation, Drama, Family, Romance]"
9430,376570,1000000,5.1912,6.731,4846,0,82,2016-03-12,"[Horror, Thriller]"


In [87]:
df2[df2['budget']==0]

,id,budget,popularity,vote_average,vote_count,revenue,runtime,release_date,genres
9401,363444,0,0.0511,5.000,1,0,90,2005-01-01,[Documentary]
8615,171982,0,0.1295,6.000,1,0,27,2012-10-09,"[Romance, Drama, Comedy]"
8307,103008,0,0.7650,2.000,1,0,85,2011-05-30,[Documentary]
9614,525662,0,0.2111,4.000,1,0,78,2018-06-22,[Documentary]
8362,111196,0,0.1792,5.200,3,0,75,2010-06-24,"[Science Fiction, Action]"
...,...,...,...,...,...,...,...,...,...
3470,11430,0,5.9532,6.580,3148,1465,77,2004-10-28,"[Adventure, Animation, Comedy, Family]"
3734,11906,0,5.4847,7.472,3180,2900000,99,1977-02-01,[Horror]
9438,378064,0,12.6228,8.387,4494,30819442,129,2016-09-17,"[Animation, Drama, Romance]"
2472,9732,0,8.0736,6.939,4673,0,81,1998-10-24,"[Adventure, Animation, Drama, Family, Romance]"


Al revisar budget, revenue y runtime se detectan valores codificados como 0 que en realidad representan datos faltantes.

De los 9620 registros de la tabla, budget tiene 3260 valores en 0 (aproximadamente un 33%) y revenue tiene 2683 valores en 0 (aproximadamente un 33%). Runtime solo tiene 1 valor en 0, por lo que se puede imputar con la mediana.

Dado el alto porcentaje de valores faltantes en budget y revenue, se decide no utilizar estas dos variables como predictoras del modelo.

In [88]:
mediana = df2['runtime'].median()
df2['runtime'] = df2['runtime'].replace(0, mediana)
df2[df2['runtime']==0]

df2 = df2[['id','popularity','vote_average','vote_count', 'runtime','release_date','genres']]
df2 = df2.rename(columns={'id': 'tmdbId','genres':'genresTMDB'})

Con las columnas de TMDB ya preparadas, se cargan las tablas de MovieLens (ratings, movies y links) y se unen entre sí. Después se combinan con los datos de TMDB a través del identificador de película, quedando así todas las valoraciones enlazadas con su información de TMDB correspondiente.

In [89]:
datosRatings = pd.read_parquet("../data/02_processed/ratings_integrity.parquet")
datosPelis = pd.read_parquet("../data/02_processed/movies_integrity.parquet")
datoslinks = pd.read_parquet("../data/02_processed/links_integrity.parquet")
totalMovieLens  = datosRatings.merge(datosPelis, on='movieId', how='left')
datosTMDB = df2.merge(datoslinks, on='tmdbId', how='left') 
datosTotal =  datosTMDB.merge(totalMovieLens, on='movieId', how='inner')
# datosTotal= datosTotal[['userId','movieId','rating','genres','year','popularity','vote_average','vote_count', 'runtime']]
datosTotal

,tmdbId,popularity,vote_average,vote_count,runtime,release_date,genresTMDB,movieId,userId,rating,timestamp,title,genres,year
0,148184,0.1573,0.0,0,56,2010-01-01,"[Documentary, History, Thriller]",152711,462,5.0,2016-02-16 18:04:22,Who Killed Chea Vichea?,Documentary,2010
1,363444,0.0511,5.0,1,90,2005-01-01,[Documentary],180777,18,4.5,2017-11-19 14:31:58,Die Frauen von Ravensbrück,Documentary,2005
2,171982,0.1295,6.0,1,27,2012-10-09,"[Romance, Drama, Comedy]",2894,325,4.0,2002-12-09 01:02:14,Romance,Drama|Romance,1999
3,103008,0.7650,2.0,1,85,2011-05-30,[Documentary],178129,599,3.5,2018-02-20 15:08:37,Adventures in Plymptoons!,Documentary,2011
4,55146,0.2646,7.0,1,113,1995-08-29,"[Drama, Thriller, Documentary, Romance]",1423,474,4.0,2009-03-29 18:16:46,Hearts and Minds,Drama,1996
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100500,157336,70.3711,8.5,39920,169,2014-11-05,"[Adventure, Drama, Science Fiction]",109487,582,4.5,2015-11-06 07:15:42,Interstellar,Sci-Fi|IMAX,2014
100501,157336,70.3711,8.5,39920,169,2014-11-05,"[Adventure, Drama, Science Fiction]",109487,596,3.5,2018-08-31 09:57:50,Interstellar,Sci-Fi|IMAX,2014
100502,157336,70.3711,8.5,39920,169,2014-11-05,"[Adventure, Drama, Science Fiction]",109487,599,3.5,2017-06-27 02:58:09,Interstellar,Sci-Fi|IMAX,2014
100503,157336,70.3711,8.5,39920,169,2014-11-05,"[Adventure, Drama, Science Fiction]",109487,601,5.0,2015-09-07 15:19:44,Interstellar,Sci-Fi|IMAX,2014


Algunas películas tienen el año vacío en MovieLens. Para esos casos se completa el año a partir de la fecha de estreno de TMDB.

In [90]:
#peliculas con año nulo-> se saca de release_date
datosTotal[datosTotal['year'].isna()]
datosTotal["release_date"] = pd.to_datetime(datosTotal["release_date"], errors="coerce")
datosTotal["year"] = datosTotal["year"].fillna(datosTotal["release_date"].dt.year)

In [91]:
datosTotal[datosTotal['year'].isna()]

,tmdbId,popularity,vote_average,vote_count,runtime,release_date,genresTMDB,movieId,userId,rating,timestamp,title,genres,year


También hay películas sin género asignado, lo que impide aplicar la codificación multi-hot directamente. En lugar de eliminar estos registros, y perder con ellos todas las valoraciones asociadas, se les asigna la categoría "sin género" (no genres listed).

In [92]:
datosTotal[datosTotal['genres'].isna()].head()

,tmdbId,popularity,vote_average,vote_count,runtime,release_date,genresTMDB,movieId,userId,rating,timestamp,title,genres,year
91,335145,0.5077,5.444,9,95,1995-11-17,[Romance],132084,599,2.5,2018-02-20 16:05:05,Let It Be Me,NaN,1995
115,55495,0.1081,6.300,10,97,1968-08-23,[Comedy],155589,89,3.0,2018-03-07 07:58:40,Noin 7 veljestä,NaN,1968
126,125464,0.2285,5.818,11,26,1977-12-04,"[Animation, Science Fiction, Family, TV Movie]",149330,514,4.0,2018-09-03 03:06:45,A Cosmic Christmas,NaN,1977
229,398854,0.6888,5.375,16,90,2016-05-30,"[Comedy, Fantasy, Romance]",159779,210,4.0,2018-07-01 17:56:02,A Midsummer Night's Dream,NaN,2016
249,49809,0.9996,5.694,18,91,1995-01-23,"[Drama, TV Movie]",181719,596,3.5,2018-09-01 20:14:07,Serving in Silence: The Margarethe Cammermeyer...,NaN,1995


In [93]:
datosTotal.loc[datosTotal['genres'].isna(), 'genres'] = '(no genres listed)'


In [94]:
genresVal = ["Action", "Adventure", "Animation", "Children", "Comedy", "Crime", "Documentary", "Drama", 
"Fantasy", "Film-Noir", "Horror", "Musical", "Mystery", "Romance", "Sci-Fi", "Thriller", "War", "Western", "IMAX",
"(no genres listed)"]
genresList = datosTotal['genres'].str.split("|")
datosTotal['genres']= genresList

Se define la lista de géneros posibles y se separa el campo genres, que llega como texto con los géneros unidos por barras, en una lista de géneros por película. Esto permite aplicar la codificación multi-hot más adelante.

In [95]:
datosTotal

,tmdbId,popularity,vote_average,vote_count,runtime,release_date,genresTMDB,movieId,userId,rating,timestamp,title,genres,year
0,148184,0.1573,0.0,0,56,2010-01-01,"[Documentary, History, Thriller]",152711,462,5.0,2016-02-16 18:04:22,Who Killed Chea Vichea?,[Documentary],2010
1,363444,0.0511,5.0,1,90,2005-01-01,[Documentary],180777,18,4.5,2017-11-19 14:31:58,Die Frauen von Ravensbrück,[Documentary],2005
2,171982,0.1295,6.0,1,27,2012-10-09,"[Romance, Drama, Comedy]",2894,325,4.0,2002-12-09 01:02:14,Romance,"[Drama, Romance]",1999
3,103008,0.7650,2.0,1,85,2011-05-30,[Documentary],178129,599,3.5,2018-02-20 15:08:37,Adventures in Plymptoons!,[Documentary],2011
4,55146,0.2646,7.0,1,113,1995-08-29,"[Drama, Thriller, Documentary, Romance]",1423,474,4.0,2009-03-29 18:16:46,Hearts and Minds,[Drama],1996
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100500,157336,70.3711,8.5,39920,169,2014-11-05,"[Adventure, Drama, Science Fiction]",109487,582,4.5,2015-11-06 07:15:42,Interstellar,"[Sci-Fi, IMAX]",2014
100501,157336,70.3711,8.5,39920,169,2014-11-05,"[Adventure, Drama, Science Fiction]",109487,596,3.5,2018-08-31 09:57:50,Interstellar,"[Sci-Fi, IMAX]",2014
100502,157336,70.3711,8.5,39920,169,2014-11-05,"[Adventure, Drama, Science Fiction]",109487,599,3.5,2017-06-27 02:58:09,Interstellar,"[Sci-Fi, IMAX]",2014
100503,157336,70.3711,8.5,39920,169,2014-11-05,"[Adventure, Drama, Science Fiction]",109487,601,5.0,2015-09-07 15:19:44,Interstellar,"[Sci-Fi, IMAX]",2014


Se seleccionan y reordenan las columnas finales que formarán parte del dataset.

In [96]:
datosTotal= datosTotal[['userId','movieId','timestamp','rating','genres','year','popularity','vote_average','vote_count', 'runtime']]

In [97]:
datosTotal

,userId,movieId,timestamp,rating,genres,year,popularity,vote_average,vote_count,runtime
0,462,152711,2016-02-16 18:04:22,5.0,[Documentary],2010,0.1573,0.0,0,56
1,18,180777,2017-11-19 14:31:58,4.5,[Documentary],2005,0.0511,5.0,1,90
2,325,2894,2002-12-09 01:02:14,4.0,"[Drama, Romance]",1999,0.1295,6.0,1,27
3,599,178129,2018-02-20 15:08:37,3.5,[Documentary],2011,0.7650,2.0,1,85
4,474,1423,2009-03-29 18:16:46,4.0,[Drama],1996,0.2646,7.0,1,113
...,...,...,...,...,...,...,...,...,...,...
100500,582,109487,2015-11-06 07:15:42,4.5,"[Sci-Fi, IMAX]",2014,70.3711,8.5,39920,169
100501,596,109487,2018-08-31 09:57:50,3.5,"[Sci-Fi, IMAX]",2014,70.3711,8.5,39920,169
100502,599,109487,2017-06-27 02:58:09,3.5,"[Sci-Fi, IMAX]",2014,70.3711,8.5,39920,169
100503,601,109487,2015-09-07 15:19:44,5.0,"[Sci-Fi, IMAX]",2014,70.3711,8.5,39920,169


In [98]:
datosTotal.info()

<class 'pandas.DataFrame'>
RangeIndex: 100505 entries, 0 to 100504
Data columns (total 10 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   userId        100505 non-null  int64         
 1   movieId       100505 non-null  int64         
 2   timestamp     100505 non-null  datetime64[ms]
 3   rating        100505 non-null  float64       
 4   genres        100505 non-null  object        
 5   year          100505 non-null  Int64         
 6   popularity    100505 non-null  float64       
 7   vote_average  100505 non-null  float64       
 8   vote_count    100505 non-null  int64         
 9   runtime       100505 non-null  int64         
dtypes: Int64(1), datetime64[ms](1), float64(3), int64(4), object(1)
memory usage: 7.8+ MB


## Codificación de géneros

Se aplica multi-hot encoding sobre la columna genres, de forma que cada género se convierte en una columna binaria independiente (1 si la película pertenece a ese género, 0 si no). Así los modelos pueden utilizar los géneros como variables numéricas.

In [99]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()

genresEncoded = mlb.fit_transform(datosTotal['genres'])

In [100]:
genresEncoded

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(100505, 20))

In [101]:
genresDf = pd.DataFrame(genresEncoded, columns=mlb.classes_)

In [102]:
datosTotal = pd.concat([datosTotal.reset_index(drop=True), genresDf], axis=1)

In [ ]:
datosTotal[['userId', 'movieId', 'timestamp', 'rating', 'year',
       'popularity', 'vote_average', 'vote_count', 'runtime',
       '(no genres listed)', 'Action', 'Adventure', 'Animation', 'Children',
       'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir',
       'Horror', 'IMAX', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller',
       'War', 'Western']]

Index(['userId', 'movieId', 'timestamp', 'rating', 'genres', 'year',
       'popularity', 'vote_average', 'vote_count', 'runtime',
       '(no genres listed)', 'Action', 'Adventure', 'Animation', 'Children',
       'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir',
       'Horror', 'IMAX', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller',
       'War', 'Western'],
      dtype='str')

Por último, se guarda el dataset resultante en formato parquet para que pueda utilizarse en el entrenamiento de los modelos híbridos.

In [104]:
datosTotal.to_parquet('../data/03_model_ready/hybrid.parquet',index=False)